# 📦 NyayaAuth — Dataset Downloader (Google Colab Version)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook downloads all IPC datasets, merges them, and saves everything to your **Google Drive** so you never lose your data between Colab sessions.

---
### ✅ Before you begin — 3 things to check:
1. You are signed into your **Google account**
2. You have a **Kaggle account** (free) at https://kaggle.com
3. Runtime → **Run all** OR run cells one by one top to bottom
---

## 🔌 Cell 1 — Mount Google Drive
This connects your Google Drive so all downloaded data is **saved permanently** (survives Colab disconnections).

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All project files will live here in your Drive
PROJECT_DIR = "/content/drive/MyDrive/NyayaAuth"

folders = [
    f"{PROJECT_DIR}/data/raw",
    f"{PROJECT_DIR}/data/raw/kaggle_downloads",
    f"{PROJECT_DIR}/data/raw/hf_downloads",
    f"{PROJECT_DIR}/data/processed",
    f"{PROJECT_DIR}/models/tfidf",
    f"{PROJECT_DIR}/models/lstm",
    f"{PROJECT_DIR}/models/tinybert",
    f"{PROJECT_DIR}/notebooks",
    f"{PROJECT_DIR}/logs",
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("✅ Google Drive mounted!")
print(f"✅ Project folder created at: {PROJECT_DIR}")
print("\nFolder structure:")
for f in folders:
    print(f"   📁 {f.replace(PROJECT_DIR, 'NyayaAuth')}")

ValueError: Mountpoint must not already contain files

## 📦 Cell 2 — Install required packages

In [ ]:
!pip install kaggle datasets pandas numpy matplotlib tqdm openpyxl -q
print("✅ All packages ready.")

## 🔑 Cell 3 — Setup Kaggle API Key

**How to get your Kaggle API key (takes 1 minute):**
1. Go to 👉 https://www.kaggle.com
2. Click your **profile picture** (top right) → **Settings**
3. Scroll down to **API** section → Click **"Create New Token"**
4. A file called `kaggle.json` will download to your computer
5. Open that file — it looks like: `{"username":"abc","key":"xyz123..."}`
6. Copy-paste your username and key into the variables below 👇

In [ ]:
import json, os

# ── PASTE YOUR KAGGLE CREDENTIALS HERE ──────────────────────
KAGGLE_USERNAME = "stuti057"     # ← paste from kaggle.json
KAGGLE_KEY      = "44dfe7eedf75d82e21a7cfe0f0aa309d"  # ← paste from kaggle.json
# ────────────────────────────────────────────────────────────

# Save credentials so Kaggle CLI can find them
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Quick test
result = os.popen("kaggle datasets list --max-size 1 2>&1").read()
if "Error" in result or "401" in result:
    print("❌ Kaggle auth failed. Double-check your username and key above.")
    print(result)
else:
    print("✅ Kaggle API connected successfully!")
    print(f"   Logged in as: {KAGGLE_USERNAME}")

## ⬇️ Cell 4 — Download Dataset 1 from Kaggle
**"Section in Indian Penal Code"** — IPC sections mapped to case descriptions

In [ ]:
import subprocess

D1_DIR = f"{PROJECT_DIR}/data/raw/kaggle_downloads/dataset1"
os.makedirs(D1_DIR, exist_ok=True)

print("Downloading Dataset 1 — masterjiii/section-in-indian-penal-code ...")
r = subprocess.run(
    ["kaggle", "datasets", "download",
     "-d", "masterjiii/section-in-indian-penal-code",
     "-p", D1_DIR, "--unzip"],
    capture_output=True, text=True
)
print(r.stdout)
if r.returncode != 0:
    print("❌", r.stderr)
else:
    print("✅ Dataset 1 downloaded!")
    print("   Files:", os.listdir(D1_DIR))

## ⬇️ Cell 5 — Download Dataset 2 from Kaggle
**"Indian Penal Code Complete Dataset"** — descriptions + punishments per IPC section

In [ ]:
D2_DIR = f"{PROJECT_DIR}/data/raw/kaggle_downloads/dataset2"
os.makedirs(D2_DIR, exist_ok=True)

print("Downloading Dataset 2 — omdabral/indian-penal-code-complete-dataset ...")
r2 = subprocess.run(
    ["kaggle", "datasets", "download",
     "-d", "omdabral/indian-penal-code-complete-dataset",
     "-p", D2_DIR, "--unzip"],
    capture_output=True, text=True
)
print(r2.stdout)
if r2.returncode != 0:
    print("❌", r2.stderr)
else:
    print("✅ Dataset 2 downloaded!")
    print("   Files:", os.listdir(D2_DIR))

## ⬇️ Cell 6 — Download Dataset 3 from Kaggle
**"Indian Crimes Dataset"** — broader Indian crime category dataset

In [ ]:
D3_DIR = f"{PROJECT_DIR}/data/raw/kaggle_downloads/dataset3"
os.makedirs(D3_DIR, exist_ok=True)

print("Downloading Dataset 3 — sudhanvahg/indian-crimes-dataset ...")
r3 = subprocess.run(
    ["kaggle", "datasets", "download",
     "-d", "sudhanvahg/indian-crimes-dataset",
     "-p", D3_DIR, "--unzip"],
    capture_output=True, text=True
)
print(r3.stdout)
if r3.returncode != 0:
    print("❌", r3.stderr)
else:
    print("✅ Dataset 3 downloaded!")
    print("   Files:", os.listdir(D3_DIR))

## ⬇️ Cell 7 — Download Dataset 4 from HuggingFace
**Dev523/Crime-Reports-Dataset** — no login needed, downloads automatically

In [ ]:
from datasets import load_dataset
import pandas as pd

HF_DIR = f"{PROJECT_DIR}/data/raw/hf_downloads"

print("Downloading from HuggingFace: Dev523/Crime-Reports-Dataset ...")
try:
    ds4 = load_dataset("Dev523/Crime-Reports-Dataset", trust_remote_code=True)
    df4 = ds4["train"].to_pandas()
    save_path = f"{HF_DIR}/crime_reports_hf.csv"
    df4.to_csv(save_path, index=False)
    print(f"✅ Downloaded! Shape: {df4.shape}")
    print(f"   Columns: {list(df4.columns)}")
    display(df4.head(3))
except Exception as e:
    print(f"⚠️  Skipped: {e}")

## ⬇️ Cell 8 — Download Dataset 5 from HuggingFace
**karan842/ipc-sections** — IPC section descriptions used to train IPC-Gemma

In [ ]:
print("Downloading from HuggingFace: karan842/ipc-sections ...")
try:
    ds5 = load_dataset("karan842/ipc-sections", trust_remote_code=True)
    parts = [ds5[split].to_pandas() for split in ds5.keys()]
    df5 = pd.concat(parts, ignore_index=True)
    save_path = f"{HF_DIR}/ipc_sections_hf.csv"
    df5.to_csv(save_path, index=False)
    print(f"✅ Downloaded! Shape: {df5.shape}")
    print(f"   Columns: {list(df5.columns)}")
    display(df5.head(3))
except Exception as e:
    print(f"⚠️  Skipped: {e}")

## 🔍 Cell 9 — Inspect ALL downloaded files
Shows every file found, its shape and column names.

In [ ]:
import pandas as pd

RAW_DIR = f"{PROJECT_DIR}/data/raw"
found_files = []

for root, dirs, files in os.walk(RAW_DIR):
    for file in files:
        if file.endswith((".csv", ".xlsx", ".json")):
            found_files.append(os.path.join(root, file))

print(f"Found {len(found_files)} file(s):\n")

for fpath in found_files:
    try:
        if fpath.endswith(".csv"):
            df_tmp = pd.read_csv(fpath)
        elif fpath.endswith(".xlsx"):
            df_tmp = pd.read_excel(fpath)
        else:
            df_tmp = pd.read_json(fpath)

        print(f"📄 {os.path.basename(fpath)}")
        print(f"   Rows   : {len(df_tmp)}")
        print(f"   Columns: {list(df_tmp.columns)}")
        print()
    except Exception as e:
        print(f"⚠️  Could not read {os.path.basename(fpath)}: {e}\n")

## 🔗 Cell 10 — Map columns & merge all datasets

> After running Cell 9, **look at the column names printed above**.
> If any column is not in `COLUMN_MAPS` below, add it before running this cell.

In [ ]:
# ── COLUMN NAME MAPPING ─────────────────────────────────────────────────────
# Add any column names printed in Cell 9 that aren't listed here
COLUMN_MAPS = {
    # Description variations
    "offense":           "description",
    "Offense":           "description",
    "crime_description": "description",
    "Description":       "description",
    "description":       "description",
    "text":              "description",
    "crime":             "description",
    "fir_text":          "description",
    "case_description":  "description",
    "details":           "description",
    "Punishment":        "description",
    "punishment":        "description",
    "offense_description": "description",

    # IPC section label variations
    "section":           "ipc_section",
    "Section":           "ipc_section",
    "ipc_section":       "ipc_section",
    "IPC_Section":       "ipc_section",
    "IPC Section":       "ipc_section",
    "ipc":               "ipc_section",
    "label":             "ipc_section",
    "category":          "ipc_section",
    "crime_type":        "ipc_section",
    "charge":            "ipc_section",
    "Chapter":           "ipc_section",
}
# ────────────────────────────────────────────────────────────────────────────

all_dfs = []

for fpath in found_files:
    try:
        if fpath.endswith(".csv"):
            df_raw = pd.read_csv(fpath)
        elif fpath.endswith(".xlsx"):
            df_raw = pd.read_excel(fpath)
        else:
            df_raw = pd.read_json(fpath)

        df_raw = df_raw.rename(columns=COLUMN_MAPS)

        if "description" in df_raw.columns and "ipc_section" in df_raw.columns:
            subset = df_raw[["description", "ipc_section"]].dropna()
            subset = subset.copy()
            subset["source"] = os.path.basename(fpath)
            all_dfs.append(subset)
            print(f"✅ {os.path.basename(fpath):45s} → {len(subset)} rows")
        else:
            print(f"⚠️  SKIPPED: {os.path.basename(fpath)}")
            print(f"   Columns found  : {list(df_raw.columns)}")
            print(f"   → Add a mapping for these columns in COLUMN_MAPS above")
    except Exception as e:
        print(f"❌ Error: {os.path.basename(fpath)}: {e}")

print(f"\nDatasets merged: {len(all_dfs)}")

## 💾 Cell 11 — Deduplicate and save final CSV to Google Drive

In [ ]:
if len(all_dfs) == 0:
    print("❌ No datasets loaded. Fix the column mappings in Cell 10 and re-run.")
else:
    merged = pd.concat(all_dfs, ignore_index=True)
    print(f"Rows before dedup : {len(merged)}")

    merged = merged.drop_duplicates(subset=["description"])
    print(f"Rows after dedup  : {len(merged)}")

    # Normalize IPC section labels
    merged["ipc_section"] = (
        merged["ipc_section"]
        .astype(str).str.strip().str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )

    out_path = f"{PROJECT_DIR}/data/raw/ipc_data.csv"
    merged.to_csv(out_path, index=False)

    print(f"\n📊 Final dataset stats:")
    print(f"   Total samples    : {len(merged)}")
    print(f"   Unique IPC sects : {merged['ipc_section'].nunique()}")
    print(f"   Sources merged   : {merged['source'].unique().tolist()}")
    print(f"\n✅ Saved to Google Drive: {out_path}")
    display(merged.head(5))

## 📊 Cell 12 — Class distribution analysis
Visualizes the imbalance — exactly as described in your paper.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

dist = merged["ipc_section"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("NyayaAuth — IPC Dataset Analysis", fontsize=14, fontweight='bold')

# Top 20 bar chart
top20 = dist.head(20)
axes[0].barh(top20.index[::-1], top20.values[::-1],
             color="steelblue", edgecolor="white")
axes[0].set_title("Top 20 IPC Sections by Sample Count")
axes[0].set_xlabel("Number of samples")
axes[0].tick_params(axis='y', labelsize=8)

# Distribution curve
axes[1].plot(range(len(dist)), dist.values, color="coral", linewidth=1.5)
axes[1].fill_between(range(len(dist)), dist.values, alpha=0.2, color="coral")
axes[1].set_title("Class Imbalance — All IPC Sections")
axes[1].set_xlabel("IPC Section rank")
axes[1].set_ylabel("Sample count")

plt.tight_layout()
plot_path = f"{PROJECT_DIR}/data/raw/class_distribution.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.show()

top10_pct = dist.head(int(len(dist)*0.1)).sum() / len(merged) * 100
print(f"\n📊 Class imbalance summary:")
print(f"   Top 10% sections cover : {top10_pct:.1f}% of all samples")
print(f"   Most common section    : {dist.index[0]}  ({dist.iloc[0]} samples)")
print(f"   Rarest section         : {dist.index[-1]}  ({dist.iloc[-1]} samples)")
print(f"   Imbalance ratio        : {dist.iloc[0] / max(dist.iloc[-1],1):.0f}:1")
print(f"\n✅ Plot saved: {plot_path}")

## ✅ Cell 13 — Final checklist

In [ ]:
checks = {
    "ipc_data.csv created"          : os.path.exists(f"{PROJECT_DIR}/data/raw/ipc_data.csv"),
    "data/processed/ folder exists" : os.path.exists(f"{PROJECT_DIR}/data/processed"),
    "class_distribution.png saved"  : os.path.exists(f"{PROJECT_DIR}/data/raw/class_distribution.png"),
    "models/ folder exists"         : os.path.exists(f"{PROJECT_DIR}/models"),
}

print("=" * 55)
print("  NyayaAuth — Dataset Download Checklist")
print("=" * 55)

for check, ok in checks.items():
    print(f"  {'✅' if ok else '❌'}  {check}")

if os.path.exists(f"{PROJECT_DIR}/data/raw/ipc_data.csv"):
    df_final = pd.read_csv(f"{PROJECT_DIR}/data/raw/ipc_data.csv")
    print(f"\n  📊 ipc_data.csv summary:")
    print(f"     Total rows      : {len(df_final)}")
    print(f"     IPC sections    : {df_final['ipc_section'].nunique()}")
    print(f"     Columns         : {list(df_final.columns)}")
    print(f"     Missing values  : {df_final.isnull().sum().to_dict()}")
    print(f"     Drive location  : {PROJECT_DIR}/data/raw/ipc_data.csv")

print("=" * 55)
print("\n🚀 Done! Next step → Run the preprocessing notebook.")
print(f"   Your data is safe in Google Drive at:")
print(f"   {PROJECT_DIR}/")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install nltk scikit-learn pandas numpy joblib -q
print("✅ Done")

In [ ]:
import pandas as pd
import numpy as np
import re, os, joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from collections import Counter

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

PROJECT_DIR = "/content/drive/MyDrive/NyayaAuth"
DATA_PATH   = f"{PROJECT_DIR}/data/raw/ipc_data.csv"
OUT_DIR     = f"{PROJECT_DIR}/data/processed"
MIN_SAMPLES = 1        # ← lowered because dataset is small
RANDOM_SEED = 42
os.makedirs(OUT_DIR, exist_ok=True)

LEGAL_KEEP = {
    "not", "no", "without", "against", "under", "above", "below",
    "between", "within", "with", "intent", "force", "death", "bodily",
    "harm", "injury", "weapon", "accused", "victim", "complainant"
}
GENERIC_STOPWORDS = set(stopwords.words("english")) - LEGAL_KEEP
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"section\s+\d+[a-z]?", "", text)
    text = re.sub(r"ipc|crpc|bns", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [lemmatizer.lemmatize(t) for t in text.split()
              if t not in GENERIC_STOPWORDS and len(t) > 1]
    return " ".join(tokens)

print("="*55)
print("  NyayaAuth — Preprocessing Pipeline (Fixed)")
print("="*55)

# ── 1. LOAD & FIX DUPLICATE COLUMNS ─────────────────────
print("\n[1/6] Loading and fixing dataset...")
df = pd.read_csv(DATA_PATH)
print(f"      Raw shape   : {df.shape}")
print(f"      All columns : {list(df.columns)}")

# Keep only the FIRST description column and ipc_section
desc_cols = [c for c in df.columns if "description" in c.lower()]
print(f"      Description columns found: {desc_cols}")
print(f"      Using: '{desc_cols[0]}' as the text column")

df = df.rename(columns={desc_cols[0]: "description"})
df = df[["description", "ipc_section"]].dropna()
df["description"]  = df["description"].astype(str).str.strip()
df["ipc_section"]  = df["ipc_section"].astype(str).str.strip().str.upper()
df = df[df["description"] != ""]
df = df[df["ipc_section"].str.len() > 0]
print(f"      After cleanup: {len(df)} rows, {df['ipc_section'].nunique()} IPC sections")

# ── 2. AUGMENT — generate more samples per IPC section ──
print("\n[2/6] Augmenting small dataset...")

# For each IPC section, create variations so we have enough to split
augmented_rows = []
for section, group in df.groupby("ipc_section"):
    for _, row in group.iterrows():
        augmented_rows.append(row)
        # Simple augmentation: shuffle sentence words slightly
        words = row["description"].split()
        if len(words) > 4:
            import random
            random.seed(42)
            # Variant 1: drop first word
            augmented_rows.append({
                "description": " ".join(words[1:]),
                "ipc_section": section
            })
            # Variant 2: drop last word
            augmented_rows.append({
                "description": " ".join(words[:-1]),
                "ipc_section": section
            })

df_aug = pd.DataFrame(augmented_rows).drop_duplicates(subset=["description"]).reset_index(drop=True)
print(f"      After augmentation: {len(df_aug)} rows")

# ── 3. CLEAN TEXT ────────────────────────────────────────
print("\n[3/6] Cleaning texts...")
df_aug["clean_text"] = df_aug["description"].apply(clean_text)
df_aug = df_aug[df_aug["clean_text"].str.strip() != ""].reset_index(drop=True)
lengths = df_aug["clean_text"].str.split().apply(len)
print(f"      Avg word count after cleaning: {lengths.mean():.1f}")

# ── 4. ENCODE LABELS ─────────────────────────────────────
print("\n[4/6] Encoding labels...")
le = LabelEncoder()
df_aug["label"] = le.fit_transform(df_aug["ipc_section"])
num_classes = len(le.classes_)
print(f"      Unique IPC sections: {num_classes}")
joblib.dump(le, f"{OUT_DIR}/label_encoder.pkl")

dist = pd.Series(Counter(df_aug["ipc_section"])).sort_values(ascending=False)
dist.to_csv(f"{OUT_DIR}/class_distribution.csv", header=["count"])

counts_arr = np.bincount(df_aug["label"], minlength=num_classes).astype(float)
counts_arr = np.where(counts_arr == 0, 1, counts_arr)
weights    = (1.0 / counts_arr)
weights    = (weights / weights.sum() * num_classes).astype(np.float32)
np.save(f"{OUT_DIR}/class_weights.npy", weights)

# ── 5. SPLIT ─────────────────────────────────────────────
print("\n[5/6] Splitting 70/15/15 (adjusted for small dataset)...")

# Use stratify only if every class has >= 2 samples
label_counts = df_aug["label"].value_counts()
can_stratify = (label_counts >= 2).all()

train_df, temp_df = train_test_split(
    df_aug, test_size=0.3,
    random_state=RANDOM_SEED,
    stratify=df_aug["label"] if can_stratify else None
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=temp_df["label"] if can_stratify else None
)
print(f"      Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ── 6. SAVE ──────────────────────────────────────────────
print("\n[6/6] Saving to Google Drive...")
train_df.to_csv(f"{OUT_DIR}/train.csv", index=False)
val_df.to_csv(f"{OUT_DIR}/val.csv",     index=False)
test_df.to_csv(f"{OUT_DIR}/test.csv",   index=False)

print("\n" + "="*55)
print("✅ Preprocessing Complete!")
print(f"   Total samples     : {len(df_aug)}")
print(f"   IPC sections      : {num_classes}")
print(f"   Train/Val/Test    : {len(train_df)}/{len(val_df)}/{len(test_df)}")
print(f"   Saved to          : {OUT_DIR}")
print("="*55)

# ── PREVIEW ──────────────────────────────────────────────
print("\nSample rows:")
display(df_aug[["description","ipc_section","clean_text","label"]].head(5))

In [ ]:
expected = [
    "train.csv", "val.csv", "test.csv",
    "label_encoder.pkl", "class_weights.npy", "class_distribution.csv"
]

print("Checking processed files:\n")
all_ok = True
for fname in expected:
    path = f"{OUT_DIR}/{fname}"
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1024 if exists else 0
    icon   = "✅" if exists else "❌"
    print(f"  {icon}  {fname:30s}  {size:.1f} KB")
    if not exists:
        all_ok = False

if all_ok:
    print("\n🚀 All good! Ready for Step 3 — Model Training.")
else:
    print("\n⚠️  Some files missing. Re-run Cell B above.")